In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from scipy.sparse import (
    csr_matrix,
    hstack,
    save_npz
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler


notes = pd.read_csv(
    "../features/preprocessed_notes.csv"
)

print("Loaded notes:", notes.shape)


Loaded notes: (10725, 9)


In [4]:
diagnoses = pd.read_csv(
    "../datasets/DIAGNOSES_ICD_sorted.csv",
    usecols=[
        "HADM_ID",
        "ICD9_CODE",
        "SEQ_NUM"
    ]
)

print("Loaded diagnoses:", diagnoses.shape)

Loaded diagnoses: (118300, 3)


In [5]:
before = len(notes)

notes = notes[
    notes["symptom_str"].notnull()
].copy()

notes = notes[
    notes["symptom_str"].str.len() > 0
].copy()

after = len(notes)

print(f"\nRemoved {before - after:,} empty rows.")


Removed 15 empty rows.


In [6]:
print("\nBuilding TF-IDF matrix...")

tfidf = TfidfVectorizer(
    min_df=5,
    max_df=0.70,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=10000
)

X_tfidf = tfidf.fit_transform(
    notes["symptom_str"]
)

print("\nTF-IDF shape:", X_tfidf.shape)


Building TF-IDF matrix...

TF-IDF shape: (10710, 10000)


In [8]:
feature_names = tfidf.get_feature_names_out()

vocab_df = pd.DataFrame({
    "feature": feature_names
})

vocab_df.to_csv(
    "../features/tfidf_vocabulary.csv",
    index=False
)

print("\nSaved TF-IDF vocabulary.")


Saved TF-IDF vocabulary.


In [9]:
tfidf_sums = np.asarray(
    X_tfidf.sum(axis=0)
).flatten()

top_idx = np.argsort(tfidf_sums)[::-1][:30]

top_terms = pd.DataFrame({
    "term": feature_names[top_idx],
    "score": tfidf_sums[top_idx]
})

print("\n=== TOP TF-IDF TERMS ===\n")

print(top_terms)


=== TOP TF-IDF TERMS ===

                    term       score
0                   pain  463.477184
1                disease  350.906586
2           hypertension  272.732217
3            respiratory  220.799742
4                failure  219.540735
5                 artery  215.941789
6                  edema  212.312665
7                  chest  208.375513
8                     of  206.924779
9              pneumonia  203.689757
10                atrial  201.636712
11            chest pain  197.225923
12          fibrillation  196.076516
13        artery disease  190.527758
14              coronary  188.477866
15       coronary artery  184.876376
16             pain pain  183.473772
17               chronic  182.930566
18                 fever  182.476232
19              diabetes  182.465168
20   atrial fibrillation  181.303330
21                 renal  179.032334
22              distress  179.018722
23                sepsis  178.179165
24  respiratory distress  177.662236
25         

In [10]:
print("\nBuilding ICD feature matrix...")

diag = diagnoses.merge(
    notes[["HADM_ID"]],
    on="HADM_ID"
)

diag = diag[
    diag["SEQ_NUM"] == 1
].copy()

TOP_N_ICD = 200

top_icd = (
    diag["ICD9_CODE"]
    .value_counts()
    .head(TOP_N_ICD)
    .index
    .tolist()
)

print(f"\nUsing top {TOP_N_ICD} ICD codes.")


Building ICD feature matrix...

Using top 200 ICD codes.


In [11]:
icd_pivot = (
    diag[
        diag["ICD9_CODE"].isin(top_icd)
    ]
    .assign(val=1)
    .pivot_table(
        index="HADM_ID",
        columns="ICD9_CODE",
        values="val",
        fill_value=0
    )
)

print("\nICD matrix shape:", icd_pivot.shape)


ICD matrix shape: (8127, 200)


In [12]:
notes_icd = (
    notes[["HADM_ID"]]
    .merge(
        icd_pivot.reset_index(),
        on="HADM_ID",
        how="left"
    )
    .fillna(0)
)

X_icd = notes_icd.drop(
    columns="HADM_ID"
).values

print("\nAligned ICD matrix:", X_icd.shape)


Aligned ICD matrix: (10710, 200)


In [13]:
X_icd_sparse = csr_matrix(X_icd)

print("\nCombining TF-IDF + ICD matrices...")

X_combined = hstack([
    X_tfidf,
    X_icd_sparse
])

print("\nCombined matrix shape:", X_combined.shape)


Combining TF-IDF + ICD matrices...

Combined matrix shape: (10710, 10200)


In [14]:
print("\nScaling features...")

scaler = MaxAbsScaler()

X_scaled = scaler.fit_transform(
    X_combined
)

print("Scaling complete.")


Scaling features...
Scaling complete.


In [15]:
save_npz(
    "../features/X_tfidf.npz",
    X_tfidf
)

save_npz(
    "../features/X_combined.npz",
    X_combined
)

save_npz(
    "../features/X_scaled.npz",
    X_scaled
)

print("\nSaved sparse matrices.")


Saved sparse matrices.


In [16]:
pd.DataFrame({
    "icd_code": top_icd
}).to_csv(
    "../features/icd_features.csv",
    index=False
)

notes[
    [
        "SUBJECT_ID",
        "HADM_ID",
        "DIAGNOSIS",
        "HOSPITAL_EXPIRE_FLAG"
    ]
].to_csv(
    "../features/patient_index.csv",
    index=False
)

print("\nSaved patient index.")


Saved patient index.


In [17]:
all_feature_names = (
    feature_names.tolist()
    +
    [f"ICD_{x}" for x in top_icd]
)

pd.DataFrame({
    "feature_name": all_feature_names
}).to_csv(
    "../features/all_feature_names.csv",
    index=False
)

print("\nSaved all feature names.")


Saved all feature names.


In [18]:

nnz = X_scaled.nnz

total = (
    X_scaled.shape[0]
    *
    X_scaled.shape[1]
)

sparsity = 1 - (nnz / total)

print("\n=== SPARSITY ===\n")

print(f"Non-zero entries: {nnz:,}")
print(f"Total entries: {total:,}")
print(f"Sparsity: {sparsity:.4f}")


=== SPARSITY ===

Non-zero entries: 537,201
Total entries: 109,242,000
Sparsity: 0.9951


In [19]:
print("\n=== FINAL SUMMARY ===\n")

print("Patients:", X_scaled.shape[0])

print("Features:", X_scaled.shape[1])

print("TF-IDF features:", X_tfidf.shape[1])

print("ICD features:", X_icd.shape[1])

print("\nFeature engineering complete.")


=== FINAL SUMMARY ===

Patients: 10710
Features: 10200
TF-IDF features: 10000
ICD features: 200

Feature engineering complete.
